<a href="https://colab.research.google.com/github/MajidSharaf/CampaignLens/blob/main/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1: running the preprocessing file on ds

In [2]:
!wget "https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments.csv"

--2026-06-01 15:32:23--  https://raw.githubusercontent.com/MajidSharaf/CampaignLens/main/Datasets/Versions/comments.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4511554 (4.3M) [text/plain]
Saving to: ‘comments.csv’

comments.csv        100%[===================>]   4.30M  --.-KB/s    in 0.02s   

2026-06-01 15:32:25 (238 MB/s) - ‘comments.csv’ saved [4511554/4511554]



In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from textblob import TextBlob
import requests
import re
import string
import nltk
import spacy

!pip install emoji langdetect contractions spacy
!python -m spacy download en_core_web_sm

import emoji
from langdetect import detect_langs
import contractions
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 34.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=8ae0b4a95170d663f0ddf8103cef3d94642903a470a73b5ab4ddaaf0521606a6
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. Yo

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_n

True

In [4]:
Trump = pd.read_csv("/content/comments.csv")
Trump = Trump[['text', 'like_count']]
Trump = Trump.dropna()
print(len(Trump))
Trump.head()

25208


,text,like_count
0,I don't want all this news only google,0
1,Trump the dumbest President in American histor...,0
2,Fareed knows full well that the reason many ob...,1
3,China's Xinjiang genocide? Millions visited Xi...,0
4,jet pilots need precision,0


In [5]:
url = "https://raw.githubusercontent.com/rishabhverma17/sms_slang_translator/master/slang.txt"
response = requests.get(url)

slang = {}
for line in response.text.splitlines():
    if '=' in line:
        key, value = line.split('=', 1)
        slang[key.strip().lower()] = value.strip().lower()

class RemoveURLs(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [re.sub(r'http\S+|www\S+', '', str(text)) for text in X]

class FixContractions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [contractions.fix(text) for text in X]

class FixRepeatedChars(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [re.sub(r'(.)\1{2,}', r'\1\1', text) for text in X]

class FixSpelling(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [str(TextBlob(text).correct()) for text in X]

class Tokenize(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [word_tokenize(text.lower()) for text in X]

class RemovePunctAndNumbers(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [[w for w in tokens if w not in string.punctuation and not w.isdigit()] for tokens in X]

class FixSlang(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for tokens in X:
            tokens = [slang.get(w, w) for w in tokens]
            tokens = [w for phrase in tokens for w in phrase.split()]
            result.append(tokens)
        return result

class HandleNegations(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for tokens in X:
            handled = []
            i = 0
            while i < len(tokens):
                if tokens[i] == 'not' and i + 1 < len(tokens):
                    handled.append(f"not_{tokens[i+1]}")
                    i += 2
                else:
                    handled.append(tokens[i])
                    i += 1
            result.append(handled)
        return result

class RemoveStopwords(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        stop_words = set(stopwords.words('english'))
        stop_words.discard('not')
        stop_words.discard('no')
        stop_words.discard('nor')
        return [[w for w in tokens if w not in stop_words] for tokens in X]

class Lemmatize(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        lemmatizer = WordNetLemmatizer()
        return [[lemmatizer.lemmatize(w) for w in tokens] for tokens in X]

class JoinTokens(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [' '.join(tokens) for tokens in X]

class DropNA(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return pd.Series(X).dropna().reset_index(drop=True).tolist()

class ConvertEmojis(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [emoji.demojize(text, delimiters=(' ', ' ')) for text in X]

class FixHashtagsAndMentions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        result = []
        for text in X:
            text = re.sub(r'@\w+', '', text)
            text = re.sub(r'#(\w+)', r'\1', text)
            result.append(text)
        return result

class KeepEnglishOnly(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = list(X)
        result = []
        for text in X:
            if not isinstance(text, str) or not text.strip():
                continue
            ascii_ratio = sum(c.isascii() for c in text) / len(text)
            if ascii_ratio >= 0.85:
                result.append(text)
        return result

class MinLengthFilter(BaseEstimator, TransformerMixin):
    def __init__(self, min_words=3):
        self.min_words = min_words
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [text for text in X if len(text.split()) >= self.min_words]

text_pipeline = Pipeline([
    ('drop_na',              DropNA()),
    ('convert_emojis',       ConvertEmojis()),
    ('fix_hashtags',         FixHashtagsAndMentions()),
    ('remove_urls',          RemoveURLs()),
    ('fix_contractions',     FixContractions()),
    ('fix_repeated',         FixRepeatedChars()),
    ('tokenize',             Tokenize()),
    ('remove_punct',         RemovePunctAndNumbers()),
    ('fix_slang',            FixSlang()),
    ('handle_negations',     HandleNegations()),
    ('remove_stopwords',     RemoveStopwords()),
    ('lemmatize',            Lemmatize()),
    ('join_tokens',          JoinTokens()),
    ('min_length',           MinLengthFilter(min_words=3)),
])

In [6]:
Trump_Processed = pd.Series(text_pipeline.fit_transform(Trump['text']))
print(len(Trump_Processed))
Trump_Processed.head()

22174


,0
0,not_want news google
1,trump dumbest president american history terri...
2,fareed know full well reason many objected jcp...
3,china 's xinjiang genocide million visited xin...
4,jet pilot need precision


In [7]:
#saving the cleaned set
Trump_Processed_df = pd.DataFrame({'cleaned_text': Trump_Processed})
Trump_Processed_df.to_csv('comments_cleaned.csv', index=False)
print("Saved! Shape:", Trump_Processed_df.shape)

Saved! Shape: (22174, 1)


 Part 2: *NER*

In [12]:
#NLTK NER
for i, comment in enumerate(Trump_Processed[:5]):
    print(f"Comment {i+1}: {comment}")
    tokens = nltk.word_tokenize(comment)
    tags = nltk.pos_tag(tokens)
    tree = nltk.ne_chunk(tags)
    print(tree)
    print()

Comment 1: not_want news google
(S not_want/JJ news/NN google/NN)

Comment 2: trump dumbest president american history terrible rollemodel america 's young people nothing violence stupidity lie
(S
  trump/NN
  dumbest/JJS
  president/NN
  american/JJ
  history/NN
  terrible/JJ
  rollemodel/NN
  america/NN
  's/POS
  young/JJ
  people/NNS
  nothing/NN
  violence/NN
  stupidity/NN
  lie/NN)

Comment 3: fareed know full well reason many objected jcpoa lifting sanction coupled sunset clause put iran world 's foremost sponsor terrorism legal funded path acquiring nuclear weapon pretend issue legitimized islamist regime disengenuous attempt lend credence embarrassment agreement outlived usefulness service no longer required
(S
  fareed/NN
  know/VBP
  full/JJ
  well/RB
  reason/NN
  many/JJ
  objected/VBD
  jcpoa/JJ
  lifting/VBG
  sanction/NN
  coupled/VBN
  sunset/NN
  clause/NN
  put/VBD
  iran/JJ
  world/NN
  's/POS
  foremost/NN
  sponsor/NN
  terrorism/NN
  legal/JJ
  funded/JJ
  path/

from output we can see NLTK didn't detect any named entities in the comments probably becuase our preprocessing lowercased all text, and NLTK relies on capitalization to identify entities like names and locations.


In [13]:
#with spaCy

nlp = spacy.load("en_core_web_sm")

for i, comment in enumerate(Trump_Processed[:5]):
    print(f"Comment {i+1}: {comment}")
    doc = nlp(comment)
    if doc.ents:
        for ent in doc.ents:
            print(f"  → {ent.text} : {ent.label_}")
    else:
        print("  -> No entities found")
    print()

Comment 1: not_want news google
  -> No entities found

Comment 2: trump dumbest president american history terrible rollemodel america 's young people nothing violence stupidity lie
  → trump dumbest : ORG
  → american : NORP
  → america : GPE

Comment 3: fareed know full well reason many objected jcpoa lifting sanction coupled sunset clause put iran world 's foremost sponsor terrorism legal funded path acquiring nuclear weapon pretend issue legitimized islamist regime disengenuous attempt lend credence embarrassment agreement outlived usefulness service no longer required
  → iran : GPE
  → islamist : NORP

Comment 4: china 's xinjiang genocide million visited xinjiang not_a single photo trace genocide lying farted zenophobia still blatantly smeared china thousand palesteninians dying hand izrale gennozide farted zealot not_blow wind
  → china : GPE
  → xinjiang : GPE
  → million : CARDINAL
  → xinjiang : GPE
  → china : GPE
  → thousand : CARDINAL

Comment 5: jet pilot need precisio

In [14]:
#Running spaCy NER on all comments
results = []

for comment in Trump_Processed:
    doc = nlp(comment)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    results.append(entities)

#Save to dataframe
NER_df = pd.DataFrame({'cleaned_text': Trump_Processed, 'entities': results})
print(NER_df.head())

                                        cleaned_text  \
0                               not_want news google   
1  trump dumbest president american history terri...   
2  fareed know full well reason many objected jcp...   
3  china 's xinjiang genocide million visited xin...   
4                           jet pilot need precision   

                                            entities  
0                                                 []  
1  [(trump dumbest, ORG), (american, NORP), (amer...  
2                    [(iran, GPE), (islamist, NORP)]  
3  [(china, GPE), (xinjiang, GPE), (million, CARD...  
4                                                 []  


In [15]:
#making sure
print(len(NER_df))

22174


In [16]:
#Most common entities
from collections import Counter
all_entities = [(ent, label) for entities in results for ent, label in entities]
entity_counts = Counter(all_entities)

#Top 20 most common entities
top_entities = pd.DataFrame(entity_counts.most_common(20), columns=['Entity', 'Count'])
print(top_entities)

                                               Entity  Count
0                                        (china, GPE)   1940
1                                    (american, NORP)   1027
2                                     (one, CARDINAL)    756
3                       (face_with_tears_of_joy, ORG)    726
4                                      (america, GPE)    707
5                                     (chinese, NORP)    546
6                                    (democrat, NORP)    484
7                                         (iran, GPE)    408
8                                  (white house, ORG)    396
9                                    (first, ORDINAL)    362
10                                       (japan, GPE)    351
11                                (thumbs_up, PERSON)    248
12                                          (xi, ORG)    242
13  (face_with_tears_of_joy face_with_tears_of_joy...    236
14                                    (two, CARDINAL)    213
15                      

In [18]:
#Save NER results
NER_df.to_csv('NER_results.csv', index=False)
print("NER results saved")

NER results saved
